# Bài 20: Giao diện web

## Từ CLI đến ứng dụng web

Bài trước đã theo dõi cách mọi thứ kết nối. Giờ là **phía người dùng** — giao diện web.

Giao diện web là **cách chính để sử dụng sản phẩm này**. Mở `http://localhost:5173` trong trình duyệt:

- Chat với team trong giao diện web gọn gàng
- Xem phản hồi streaming (hiển thị theo thời gian thực)
- Duyệt và đọc các bài viết đã tạo từ thanh bên (sidebar)
- Xóa các bài viết không cần

Kiến trúc: **FastAPI backend** (AgentOS) + **React frontend** (Vite).

## AgentOS — Backend trong 30 dòng

Agno cung cấp **AgentOS** — một wrapper biến bất kỳ Agent hoặc Team nào thành một FastAPI server với hơn 50 endpoint.

```python
from agno.os import AgentOS
from agents.team import team

agent_os = AgentOS(teams=[team])
app = agent_os.get_app()
```

Chỉ vậy thôi. AgentOS tự động tạo:
- `POST /teams/seo-workspace/runs` — gửi prompt (hỗ trợ SSE streaming)
- `GET /health` — kiểm tra sức khỏe server
- `GET /docs` — tài liệu API tương tác (Swagger)
- Quản lý session, lịch sử chạy, và nhiều hơn nữa

File `serve.py` của chúng ta thêm các route tùy chỉnh cho tầng lưu trữ bài viết (article storage) lên trên đó.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Let's read the actual serve.py
serve_path = os.path.abspath("../../output/backend/serve.py")
with open(serve_path, "r", encoding="utf-8") as f:
    print(f.read())

## Phân tích serve.py

File gồm 4 phần:

1. **Xác thực API key** — giống như chat.py, thoát nếu thiếu ANTHROPIC_API_KEY
2. **FastAPI app tùy chỉnh** — với CORS middleware (cho phép React frontend kết nối) và 3 route cho bài viết
3. **AgentOS wrapper** — `AgentOS(teams=[team], base_app=app)` gộp các route tùy chỉnh với các route tự sinh
4. **Khởi chạy server** — `agent_os.serve(app="serve:app", port=7777, reload=True)`

Các route bài viết tùy chỉnh là wrapper mỏng quanh các hàm trong `tools.storage`:
- `GET /api/articles` → gọi `list_articles()`
- `GET /api/articles/{id}` → gọi `get_article()`
- `DELETE /api/articles/{id}` → xóa khỏi metadata + xóa file `.md`

## Frontend

React frontend nằm trong thư mục `frontend/` ở gốc dự án:

```
frontend/
├── package.json          Dependencies (react, react-markdown, vite)
├── vite.config.js        Dev server proxy → localhost:7777
├── index.html
└── src/
    ├── main.jsx          Điểm vào ứng dụng
    ├── App.jsx           Layout: sidebar + vùng chính
    ├── api.js            API client (các hàm fetch)
    └── components/
        ├── Chat.jsx      Giao diện chat với SSE streaming
        ├── ArticleList.jsx  Danh sách bài viết trong sidebar
        └── ArticleView.jsx  Trang xem bài viết đầy đủ
```

**SSE (Server-Sent Events)** là cách frontend nhận phản hồi streaming. Thay vì chờ toàn bộ phản hồi, trình duyệt nhận từng phần khi chúng được tạo ra — giống trải nghiệm như ChatGPT.

Vite dev server proxy các request `/api` và `/teams` đến `localhost:7777` (backend), nên frontend chỉ cần dùng URL tương đối.

## Chạy ứng dụng web

Cần hai terminal:

**Terminal 1 — Backend:**
```bash
python output/backend/serve.py
```
Khởi chạy trên `http://localhost:7777`. Tài liệu Swagger tại `/docs`.

**Terminal 2 — Frontend:**
```bash
cd output/frontend
npm install    # chỉ lần đầu
npm run dev
```
Khởi chạy trên `http://localhost:5173`. Mở trong trình duyệt.

**Thay thế bằng CLI:**
```bash
python output/backend/chat.py
```
Cùng team, giao diện terminal. Hữu ích cho việc phát triển và kiểm thử.

In [ ]:
print("Để chạy ứng dụng web:\n")
print("  Terminal 1 (backend):  python output/backend/serve.py")
print("  Terminal 2 (frontend): cd output/frontend && npm run dev")
print()
print("Backend: http://localhost:7777")
print("  /docs      -- Tài liệu Swagger API")
print("  /health    -- Kiểm tra sức khỏe server")
print("  /api/articles -- Danh sách bài viết")
print()
print("Frontend: http://localhost:5173")
print("  Chat với team, duyệt bài viết")
print()
print("Thay thế bằng CLI: python output/backend/chat.py")

## Vibecoding backend

Đây là cách `serve.py` được tạo ra — **không phải viết tay, mà bằng cách mô tả cho Claude Code**.

Quy trình:
1. File `CLAUDE.md` đã mô tả toàn bộ kiến trúc (agents, tools, storage)
2. Chúng ta bảo Claude Code: *"Tạo một FastAPI backend dùng AgentOS để phục vụ team, với các route tùy chỉnh cho tầng lưu trữ bài viết"*
3. Claude Code đọc code hiện có, hiểu các pattern, và tạo ra `serve.py`

Điểm mấu chốt: **bạn không cần biết cú pháp FastAPI hay AgentOS**. Bạn mô tả những gì bạn muốn dựa trên code đã có, và Claude Code tạo ra phần triển khai.

Lý do nó hoạt động tốt:
- **CLAUDE.md** cho Claude Code đầy đủ ngữ cảnh về dự án
- **Code hiện có** (team.py, storage.py) sạch và có tổ chức
- **Prompt** cụ thể: *team nào, route nào, port nào*

In [ ]:
# Let's see the CLAUDE.md that gives Claude Code context
claude_md_path = os.path.abspath("../../CLAUDE.md")
with open(claude_md_path, "r", encoding="utf-8") as f:
    content = f.read()

print(f"CLAUDE.md ({len(content.splitlines())} dòng)\n")
print(content[:3000])
if len(content) > 3000:
    print("\n... (lược bớt)")

## Ghi chú: json.loads() và json.dumps()

Bạn sẽ thấy hai hàm này khi làm việc với dữ liệu JSON trong Python:

- `json.loads(text)` — **load string**: chuyển chuỗi JSON thành dict/list Python
- `json.dumps(data)` — **dump string**: chuyển dict/list Python thành chuỗi JSON

```python
import json

# Chuỗi JSON -> dict Python
text = '{"title": "SEO Guide", "words": 1500}'
data = json.loads(text)
print(data["title"])  # "SEO Guide"

# Dict Python -> chuỗi JSON
back_to_text = json.dumps(data)
print(back_to_text)   # '{"title": "SEO Guide", "words": 1500}'
```

Chữ 's' trong loads/dumps là viết tắt của 'string'. Ngoài ra còn có `json.load(file)` / `json.dump(data, file)` để đọc/ghi file trực tiếp.

## Đọc code sản phẩm

### Bản đồ: Bài học → File sản phẩm

| Bạn đã học | Bài học | File sản phẩm |
|---|---|---|
| Cách LLM hoạt động, prompt, model | 05-07 | (Kiến thức nền — định hướng mọi quyết định) |
| Tạo agent, tools | 08-09 | `output/backend/agents/content_writer.py`, `image_finder.py`, `aio_analyzer.py` |
| Structured output (Pydantic) | 10-11 | (Dùng trong phân tích AIO) |
| Nối chuỗi agent, mini pipeline | 12-13 | (Khái niệm — thay bằng team delegation) |
| Content Writer, Images, AIO | 15-16 | `output/backend/agents/content_writer.py`, `image_finder.py`, `aio_analyzer.py` |
| Team và xử lý hàng loạt | 17 | `output/backend/agents/team.py` |
| Lưu trữ file cục bộ | 18 | `output/backend/tools/storage.py` |
| Cách mọi thứ kết nối | 19 | Tất cả file — toàn bộ luồng yêu cầu |
| Giao diện web (bài này) | 20 | `output/backend/serve.py` + `frontend/` |

### Cấu trúc file

```
output/
├── chat.py                     Điểm vào CLI
├── serve.py                    Backend web (AgentOS + route bài viết)
├── agents/                     Định nghĩa agent (ai làm gì)
│   ├── __init__.py             Re-exports
│   ├── content_writer.py       Content Writer
│   ├── image_finder.py         Image Finder
│   ├── aio_analyzer.py         AIO Analyzer
│   └── team.py                 Agno Team
└── tools/                      Định nghĩa tool (làm gì được)
    ├── __init__.py             Package marker
    ├── storage.py              Lưu trữ file cục bộ
    ├── search.py               DataForSEO web search
    ├── aio.py                  Phân tích AIO + credentials
    └── images.py               DataForSEO image search
```

## Tổng kết

Sản phẩm có **hai giao diện** dùng chung một team:

| | Giao diện web | CLI |
|---|---|---|
| File | `serve.py` | `chat.py` |
| Framework | FastAPI + AgentOS | Agno cli_app() |
| Truy cập | Trình duyệt (localhost:5173) | Terminal |
| Streaming | SSE (Server-Sent Events) | Tích hợp sẵn |
| Bài viết | Duyệt trong sidebar | JSON trong terminal |
| Phù hợp | Sử dụng hàng ngày | Phát triển/kiểm thử |

Cả hai dùng **cùng team, cùng agent, cùng tool, cùng storage**.

**Bài tiếp theo:** Mở rộng sản phẩm — thêm khả năng mới bằng vibecoding.

## Bài tập

1. Đọc `output/backend/serve.py` và xác định: có bao nhiêu route tùy chỉnh? Mỗi route làm gì?
2. File `api.js` của frontend xử lý SSE streaming. Nó gọi endpoint nào để chat? (Gợi ý: tìm hàm `runTeam`)
3. Nếu bạn muốn thêm route `PUT /api/articles/{id}` để cập nhật metadata bài viết, hàm storage nào sẽ được gọi?

<details>
<summary>Bấm để xem đáp án</summary>

1. **3 route tùy chỉnh:** `GET /api/articles` (liệt kê tất cả), `GET /api/articles/{id}` (lấy một bài), `DELETE /api/articles/{id}` (xóa một bài).

2. **Endpoint chat:** `POST /teams/seo-workspace/runs` với `stream: true` trong body request. Frontend đọc các SSE event từ response.

3. **Route PUT:** Nó sẽ gọi `update_article_content(article_id, article_markdown)` từ `tools.storage`. Bạn cũng cần cập nhật các trường metadata trong `articles.json` trực tiếp hoặc thêm một hàm storage mới.
</details>